In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import re
import shutil
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

BASE_OUTPUT_DIR = "/content/drive/MyDrive/Weather Trend Forecasting/processed_outputs"

INPUT_CLEANED_FILE = os.path.join(
    BASE_OUTPUT_DIR,
    "GlobalWeatherRepository_missing_cleaned.csv"
)

INPUT_CITY_DIR = os.path.join(
    BASE_OUTPUT_DIR,
    "cleaned_by_city_strict"
)

OUTLIER_OUTPUT_DIR = os.path.join(
    BASE_OUTPUT_DIR,
    "outlier_processed_outputs"
)

OUTLIER_CITY_DIR = os.path.join(
    OUTLIER_OUTPUT_DIR,
    "outlier_cleaned_by_city"
)

FIGURE_DIR = os.path.join(
    OUTLIER_OUTPUT_DIR,
    "outlier_figures"
)

REPORT_DIR = os.path.join(
    OUTLIER_OUTPUT_DIR,
    "outlier_reports"
)

for path in [OUTLIER_OUTPUT_DIR, OUTLIER_CITY_DIR, FIGURE_DIR, REPORT_DIR]:
    os.makedirs(path, exist_ok=True)

df = pd.read_csv(INPUT_CLEANED_FILE)
df["last_updated"] = pd.to_datetime(df["last_updated"], errors="coerce")

df = df.dropna(subset=["country", "location_name", "last_updated"])
df = df.sort_values(["location_name", "last_updated"]).reset_index(drop=True)

print("Loaded cleaned dataset shape:", df.shape)
display(df.head())

Mounted at /content/drive
Loaded cleaned dataset shape: (141508, 41)


,last_updated,country,location_name,latitude,longitude,timezone,last_updated_epoch,temperature_celsius,temperature_fahrenheit,condition_text,...,air_quality_PM2.5,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination
0,2024-05-31 16:15:00,Belgium,'S Gravenjansdijk,51.25,3.63,Europe/Brussels,1717164900,16.0,60.8,Moderate rain,...,2.2,3.4,1,1,05:36 AM,09:52 PM,02:59 AM,02:09 PM,Waning Crescent,47
1,2024-06-01 16:30:00,Belgium,'S Gravenjansdijk,51.25,3.63,Europe/Brussels,1717252200,16.0,60.8,Overcast,...,6.5,19.6,1,1,05:35 AM,09:53 PM,03:12 AM,03:34 PM,Waning Crescent,35
2,2024-06-04 16:15:00,Belgium,'S Gravenjansdijk,51.25,3.63,Europe/Brussels,1717510500,18.0,64.4,Partly cloudy,...,5.7,6.0,1,1,05:33 AM,09:56 PM,03:56 AM,07:57 PM,Waning Crescent,8
3,2024-06-05 16:15:00,Belgium,'S Gravenjansdijk,51.25,3.63,Europe/Brussels,1717596900,15.0,59.0,Partly cloudy,...,1.2,2.4,1,1,05:33 AM,09:56 PM,04:17 AM,09:25 PM,Waning Crescent,3
4,2024-06-11 16:15:00,Belgium,'S Gravenjansdijk,51.25,3.63,Europe/Brussels,1718115300,15.2,59.4,Partly cloudy,...,0.5,0.9,1,1,05:30 AM,10:01 PM,10:16 AM,01:31 AM,Waxing Crescent,21


In [2]:
# Select numeric features for outlier detection

exclude_numeric_cols = [
    "last_updated_epoch",
    "latitude",
    "longitude"
]

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

outlier_numeric_cols = [
    col for col in numeric_cols
    if col not in exclude_numeric_cols
]

print("Numeric features for outlier detection:")
print(outlier_numeric_cols)
print("Number of selected features:", len(outlier_numeric_cols))

Numeric features for outlier detection:
['temperature_celsius', 'temperature_fahrenheit', 'wind_mph', 'wind_kph', 'wind_degree', 'pressure_mb', 'pressure_in', 'precip_mm', 'precip_in', 'humidity', 'cloud', 'feels_like_celsius', 'feels_like_fahrenheit', 'visibility_km', 'visibility_miles', 'uv_index', 'gust_mph', 'gust_kph', 'air_quality_Carbon_Monoxide', 'air_quality_Ozone', 'air_quality_Nitrogen_dioxide', 'air_quality_Sulphur_dioxide', 'air_quality_PM2.5', 'air_quality_PM10', 'air_quality_us-epa-index', 'air_quality_gb-defra-index', 'moon_illumination']
Number of selected features: 27


In [3]:
# Only values that are physically or definitionally impossible are marked as invalid.

physical_ranges = {
    "humidity": (0, 100),
    "cloud": (0, 100),
    "moon_illumination": (0, 100),
    "wind_degree": (0, 360),

    "uv_index": (0, None),
    "visibility_km": (0, None),
    "visibility_miles": (0, None),
    "wind_mph": (0, None),
    "wind_kph": (0, None),
    "gust_mph": (0, None),
    "gust_kph": (0, None),
    "precip_mm": (0, None),
    "precip_in": (0, None),
    "air_quality_Carbon_Monoxide": (0, None),
    "air_quality_Ozone": (0, None),
    "air_quality_Nitrogen_dioxide": (0, None),
    "air_quality_Sulphur_dioxide": (0, None),
    "air_quality_PM2.5": (0, None),
    "air_quality_PM10": (0, None),

    "air_quality_us-epa-index": (1, 6),
    "air_quality_gb-defra-index": (1, 10),
}


df_physical = df.copy()
physical_invalid_records = []
invalid_detail_records = []

for col, bounds in physical_ranges.items():
    if col not in df_physical.columns:
        continue
    lower, upper = bounds

    # Convert to numeric safely
    df_physical[col] = pd.to_numeric(df_physical[col], errors="coerce")
    invalid_mask = pd.Series(False, index=df_physical.index)
    if lower is not None:
        invalid_mask |= df_physical[col] < lower
    if upper is not None:
        invalid_mask |= df_physical[col] > upper
    invalid_count = int(invalid_mask.sum())
    physical_invalid_records.append({
        "feature": col,
        "lower_bound": lower,
        "upper_bound": upper,
        "invalid_count": invalid_count,
        "invalid_rate_%": invalid_count / len(df_physical) * 100
    })

    if invalid_count > 0:
        invalid_rows = df_physical.loc[
            invalid_mask,
            ["country", "location_name", "last_updated", col]
        ].copy()
        invalid_rows = invalid_rows.rename(columns={col: "invalid_value"})
        invalid_rows["feature"] = col
        invalid_rows["lower_bound"] = lower
        invalid_rows["upper_bound"] = upper
        invalid_detail_records.append(invalid_rows)

        # Mark invalid values as NaN for later city-wise temporal repair
        df_physical.loc[invalid_mask, col] = np.nan

physical_invalid_summary = pd.DataFrame(physical_invalid_records)
physical_invalid_summary = physical_invalid_summary.sort_values(
    "invalid_count",
    ascending=False
)

display(physical_invalid_summary)
physical_invalid_summary.to_csv(
    os.path.join(REPORT_DIR, "physical_invalid_value_summary.csv"),
    index=False
)


if len(invalid_detail_records) > 0:
    physical_invalid_details = pd.concat(
        invalid_detail_records,
        ignore_index=True
    )
    display(physical_invalid_details.head(30))
    physical_invalid_details.to_csv(
        os.path.join(REPORT_DIR, "physical_invalid_value_details.csv"),
        index=False
    )
else:
    physical_invalid_details = pd.DataFrame()
    print("No physically invalid values were detected.")

,feature,lower_bound,upper_bound,invalid_count,invalid_rate_%
18,air_quality_PM10,0,NaN,2,0.001413
13,air_quality_Carbon_Monoxide,0,NaN,1,0.000707
16,air_quality_Sulphur_dioxide,0,NaN,1,0.000707
1,cloud,0,100.0,0,0.000000
0,humidity,0,100.0,0,0.000000
4,uv_index,0,NaN,0,0.000000
3,wind_degree,0,360.0,0,0.000000
2,moon_illumination,0,100.0,0,0.000000
5,visibility_km,0,NaN,0,0.000000
9,gust_mph,0,NaN,0,0.000000


,country,location_name,last_updated,invalid_value,feature,lower_bound,upper_bound
0,Antigua and Barbuda,Saint John's,2024-06-11 10:00:00,-9999.00,air_quality_Carbon_Monoxide,0,None
1,Kiribati,Tarawa,2024-07-17 00:45:00,-9999.00,air_quality_Sulphur_dioxide,0,None
2,Saudi Arabia,Riyadh,2025-02-25 13:00:00,-1848.15,air_quality_PM10,0,None
3,Saudi Arabia,Riyadh,2026-01-25 10:00:00,-998.15,air_quality_PM10,0,None


In [17]:
# Repair physically invalid values using city-wise time interpolation with detailed modification log
df_physical["last_updated"] = pd.to_datetime(df_physical["last_updated"], errors="coerce")

# Build invalid-value log BEFORE interpolation
if "physical_invalid_details" in globals() and len(physical_invalid_details) > 0:
    invalid_value_log = physical_invalid_details.copy()
    invalid_value_log["last_updated"] = pd.to_datetime(
        invalid_value_log["last_updated"],
        errors="coerce"
    )
    print("Number of physically invalid values marked as NaN:")
    print(len(invalid_value_log))
else:
    invalid_value_log = pd.DataFrame(
        columns=[
            "country",
            "location_name",
            "last_updated",
            "invalid_value",
            "feature",
            "lower_bound",
            "upper_bound"
        ]
    )
    print("No physically invalid values were detected before interpolation.")

# Interpolation function
def interpolate_city_numeric(group, cols, max_gap=6):
    group = group.copy()
    group = group.sort_values("last_updated")
    group = group.set_index("last_updated")
    for col in cols:
        if col in group.columns:
            group[col] = pd.to_numeric(group[col], errors="coerce")
            group[col] = group[col].interpolate(
                method="time",
                limit=max_gap,
                limit_direction="both"
            )
    group = group.reset_index()
    return group

# Apply city-wise interpolation
df_physical_repaired = (
    df_physical
    .groupby("location_name", group_keys=False)
    .apply(lambda g: interpolate_city_numeric(g, outlier_numeric_cols, max_gap=6))
)
print("After physical validation and repair:", df_physical_repaired.shape)

# Build repair log: original invalid value -> repaired value
repair_records = []
if len(invalid_value_log) > 0:
    lookup_cols = ["country", "location_name", "last_updated"] + outlier_numeric_cols
    repaired_lookup = df_physical_repaired[lookup_cols].copy()
    repaired_lookup["last_updated"] = pd.to_datetime(
        repaired_lookup["last_updated"],
        errors="coerce"
    )
    for _, row in invalid_value_log.iterrows():
        country = row["country"]
        city = row["location_name"]
        timestamp = row["last_updated"]
        feature = row["feature"]
        original_value = row["invalid_value"]

        matched = repaired_lookup[
            (repaired_lookup["country"] == country) &
            (repaired_lookup["location_name"] == city) &
            (repaired_lookup["last_updated"] == timestamp)
        ]
        if len(matched) == 0:
            repaired_value = np.nan
            repair_status = "row_not_found_after_interpolation"
        else:
            repaired_value = matched.iloc[0][feature]
            if pd.isna(repaired_value):
                repair_status = "not_repaired_remaining_nan"
            else:
                repair_status = "repaired_by_city_time_interpolation"

        repair_records.append({
            "country": country,
            "location_name": city,
            "last_updated": timestamp,
            "feature": feature,
            "original_invalid_value": original_value,
            "temporary_value": np.nan,
            "repaired_value": repaired_value,
            "lower_bound": row.get("lower_bound", np.nan),
            "upper_bound": row.get("upper_bound", np.nan),
            "repair_status": repair_status
        })

physical_repair_log = pd.DataFrame(repair_records)
display(physical_repair_log.head(30))

# Summary: how many values and rows were modified
if len(physical_repair_log) > 0:
    total_invalid_values = len(physical_repair_log)
    successfully_repaired_values = (
        physical_repair_log["repair_status"] == "repaired_by_city_time_interpolation"
    ).sum()
    unrepaired_values = (
        physical_repair_log["repair_status"] != "repaired_by_city_time_interpolation"
    ).sum()
    modified_rows = (
        physical_repair_log[
            physical_repair_log["repair_status"] == "repaired_by_city_time_interpolation"
        ][["country", "location_name", "last_updated"]]
        .drop_duplicates()
        .shape[0]
    )
    repair_summary = pd.DataFrame([{
        "total_physically_invalid_values_detected": total_invalid_values,
        "successfully_repaired_values": successfully_repaired_values,
        "unrepaired_values_remaining_nan": unrepaired_values,
        "number_of_rows_modified": modified_rows
    }])
else:
    repair_summary = pd.DataFrame([{
        "total_physically_invalid_values_detected": 0,
        "successfully_repaired_values": 0,
        "unrepaired_values_remaining_nan": 0,
        "number_of_rows_modified": 0
    }])
display(repair_summary)

# Save repair log and summary
physical_repair_log.to_csv(
    os.path.join(REPORT_DIR, "physical_invalid_value_repair_log.csv"),
    index=False
)
repair_summary.to_csv(
    os.path.join(REPORT_DIR, "physical_invalid_value_repair_summary.csv"),
    index=False
)
print("Saved physical invalid value repair log to:")
print(os.path.join(REPORT_DIR, "physical_invalid_value_repair_log.csv"))
print("Saved physical invalid value repair summary to:")
print(os.path.join(REPORT_DIR, "physical_invalid_value_repair_summary.csv"))

df_physical = df_physical_repaired.copy()

Number of physically invalid values marked as NaN:
4
After physical validation and repair: (141508, 41)


,country,location_name,last_updated,feature,original_invalid_value,temporary_value,repaired_value,lower_bound,upper_bound,repair_status
0,Antigua and Barbuda,Saint John's,2024-06-11 10:00:00,air_quality_Carbon_Monoxide,-9999.00,NaN,188.669792,0,None,repaired_by_city_time_interpolation
1,Kiribati,Tarawa,2024-07-17 00:45:00,air_quality_Sulphur_dioxide,-9999.00,NaN,0.050262,0,None,repaired_by_city_time_interpolation
2,Saudi Arabia,Riyadh,2025-02-25 13:00:00,air_quality_PM10,-1848.15,NaN,2463.830000,0,None,repaired_by_city_time_interpolation
3,Saudi Arabia,Riyadh,2026-01-25 10:00:00,air_quality_PM10,-998.15,NaN,1831.164660,0,None,repaired_by_city_time_interpolation


,total_physically_invalid_values_detected,successfully_repaired_values,unrepaired_values_remaining_nan,number_of_rows_modified
0,4,4,0,4


Saved physical invalid value repair log to:
/content/drive/MyDrive/Weather Trend Forecasting/processed_outputs/outlier_processed_outputs/outlier_reports/physical_invalid_value_repair_log.csv
Saved physical invalid value repair summary to:
/content/drive/MyDrive/Weather Trend Forecasting/processed_outputs/outlier_processed_outputs/outlier_reports/physical_invalid_value_repair_summary.csv


In [5]:
# Save Physical-Validated Dataset
import unicodedata

def clean_city_filename(city_name):
    city_name = str(city_name)
    city_name = unicodedata.normalize("NFKD", city_name)
    city_name = city_name.encode("ascii", "ignore").decode("ascii")
    city_name = re.sub(r"[^A-Za-z0-9]+", "_", city_name)
    city_name = re.sub(r"_+", "_", city_name)
    city_name = city_name.strip("_")
    if city_name == "":
        city_name = "unknown_city"
    return city_name

PHYSICAL_VALIDATED_FILE = os.path.join(
    OUTLIER_OUTPUT_DIR,
    "GlobalWeatherRepository_physical_validated.csv"
)

PHYSICAL_CITY_DIR = os.path.join(
    OUTLIER_OUTPUT_DIR,
    "physical_validated_by_city"
)

os.makedirs(PHYSICAL_CITY_DIR, exist_ok=True)
df_physical = df_physical.sort_values(["location_name", "last_updated"]).reset_index(drop=True)
df_physical.to_csv(PHYSICAL_VALIDATED_FILE, index=False)
used_filenames = set()

for city, g in df_physical.groupby("location_name"):
    safe_city_name = clean_city_filename(city)
    city_file = os.path.join(PHYSICAL_CITY_DIR, f"{safe_city_name}_physical_validated.csv")
    counter = 1
    while city_file in used_filenames or os.path.exists(city_file):
        city_file = os.path.join(PHYSICAL_CITY_DIR, f"{safe_city_name}_{counter}_physical_validated.csv")
        counter += 1
    used_filenames.add(city_file)
    g.sort_values("last_updated").to_csv(city_file, index=False)

print("Saved physical-validated full dataset to:")
print(PHYSICAL_VALIDATED_FILE)
print("Saved physical-validated city files to:")
print(PHYSICAL_CITY_DIR)

Saved physical-validated full dataset to:
/content/drive/MyDrive/Weather Trend Forecasting/processed_outputs/outlier_processed_outputs/GlobalWeatherRepository_physical_validated.csv
Saved physical-validated city files to:
/content/drive/MyDrive/Weather Trend Forecasting/processed_outputs/outlier_processed_outputs/physical_validated_by_city


In [6]:
# City-Level IQR, Z-score, Isolation Forest Analysis
# No numerical modification in this step

STAT_REPORT_DIR = os.path.join(REPORT_DIR, "city_level_statistical_outliers")
STAT_FIGURE_DIR = os.path.join(FIGURE_DIR, "city_level_statistical_outliers")

os.makedirs(STAT_REPORT_DIR, exist_ok=True)
os.makedirs(STAT_FIGURE_DIR, exist_ok=True)

df_stat = df_physical.copy()
df_stat = df_stat.sort_values(["location_name", "last_updated"]).reset_index(drop=True)

iqr_flag_cols = []
zscore_flag_cols = []

for col in outlier_numeric_cols:
    df_stat[f"{col}_iqr_outlier"] = False
    df_stat[f"{col}_zscore_outlier"] = False
    iqr_flag_cols.append(f"{col}_iqr_outlier")
    zscore_flag_cols.append(f"{col}_zscore_outlier")

df_stat["isolation_forest_outlier"] = False
df_stat["isolation_forest_score"] = np.nan

city_stat_summary_records = []
for city, g in df_stat.groupby("location_name"):
    g = g.sort_values("last_updated").copy()
    city_idx = g.index
    safe_city_name = clean_city_filename(city)

    # IQR and Z-score per city
    for col in outlier_numeric_cols:
        if col not in g.columns:
            continue
        series = pd.to_numeric(g[col], errors="coerce")
        if series.notna().sum() < 10:
            continue

        # IQR
        Q1 = series.quantile(0.25)
        Q3 = series.quantile(0.75)
        IQR = Q3 - Q1
        if IQR > 0:
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            iqr_mask = (series < lower_bound) | (series > upper_bound)
            df_stat.loc[city_idx, f"{col}_iqr_outlier"] = iqr_mask.values

        # Z-score
        mean_val = series.mean()
        std_val = series.std()
        if std_val > 0:
            z = (series - mean_val) / std_val
            z_mask = z.abs() > 3
            df_stat.loc[city_idx, f"{col}_zscore_outlier"] = z_mask.values

    # Isolation Forest per city
    iso_features = [
        col for col in outlier_numeric_cols
        if col in g.columns and g[col].notna().sum() >= 30 and g[col].std() > 0]
    if len(g) >= 50 and len(iso_features) >= 3:
        iso_data = g[iso_features].copy()
        iso_data = iso_data.fillna(iso_data.median(numeric_only=True))
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(iso_data)
        iso_model = IsolationForest(
            n_estimators=200,
            contamination=0.03,
            random_state=42,
            n_jobs=-1
        )
        iso_pred = iso_model.fit_predict(X_scaled)
        iso_score = iso_model.decision_function(X_scaled)
        df_stat.loc[city_idx, "isolation_forest_outlier"] = iso_pred == -1
        df_stat.loc[city_idx, "isolation_forest_score"] = iso_score

    # City summary
    iqr_count = int(df_stat.loc[city_idx, iqr_flag_cols].sum().sum())
    zscore_count = int(df_stat.loc[city_idx, zscore_flag_cols].sum().sum())
    iso_count = int(df_stat.loc[city_idx, "isolation_forest_outlier"].sum())
    city_stat_summary_records.append({
        "location_name": city,
        "country": g["country"].iloc[0],
        "record_count": len(g),
        "iqr_outlier_value_count": iqr_count,
        "zscore_outlier_value_count": zscore_count,
        "isolation_forest_row_count": iso_count
    })

    # Save per-city statistical flags
    flag_cols = (
        ["country", "location_name", "last_updated"]
        + iqr_flag_cols
        + zscore_flag_cols
        + ["isolation_forest_outlier", "isolation_forest_score"]
    )
    city_flag_file = os.path.join(
        STAT_REPORT_DIR,
        f"{safe_city_name}_statistical_outlier_flags.csv"
    )
    df_stat.loc[city_idx, flag_cols].to_csv(city_flag_file, index=False)

    # Save simple visualization per city
    key_plot_features = [
        "temperature_celsius",
        "humidity",
        "pressure_mb",
        "wind_kph",
        "gust_kph",
        "precip_mm",
        "air_quality_PM2.5",
        "air_quality_PM10"
    ]

    key_plot_features = [c for c in key_plot_features if c in g.columns]
    if len(key_plot_features) > 0:
        fig, axes = plt.subplots(
            nrows=len(key_plot_features),
            ncols=1,
            figsize=(12, 3 * len(key_plot_features))
        )
        if len(key_plot_features) == 1:
            axes = [axes]
        for ax, col in zip(axes, key_plot_features):
            ax.plot(g["last_updated"], g[col], linewidth=1)
            ax.set_title(f"{city} - {col}")
            ax.set_xlabel("Time")
            ax.set_ylabel(col)
        plt.tight_layout()
        fig_path = os.path.join(
            STAT_FIGURE_DIR,
            f"{safe_city_name}_key_feature_timeseries.png"
        )
        plt.savefig(fig_path, dpi=200)
        plt.close()

city_stat_summary = pd.DataFrame(city_stat_summary_records)
city_stat_summary = city_stat_summary.sort_values(
    ["iqr_outlier_value_count", "zscore_outlier_value_count", "isolation_forest_row_count"],
    ascending=False
)
display(city_stat_summary.head(30))
city_stat_summary.to_csv(
    os.path.join(REPORT_DIR, "city_level_statistical_outlier_summary.csv"),
    index=False
)
df_stat.to_csv(
    os.path.join(OUTLIER_OUTPUT_DIR, "GlobalWeatherRepository_statistical_flags_only.csv"),
    index=False
)

print("Saved city-level statistical outlier analysis.")

,location_name,country,record_count,iqr_outlier_value_count,zscore_outlier_value_count,isolation_forest_row_count
175,Paramaribo,Suriname,724,1413,223,22
89,Guatemala City,Guatemala,720,1238,304,22
49,Bras,Brazil,722,1086,167,22
174,Panama City,Panama,724,1015,215,22
158,National,Bolivia,722,939,337,22
106,Kigali,Rwanda,727,873,199,22
184,Port Of Spain,Trinidad and Tobago,724,832,252,22
86,Georgetown,Guyana,723,830,237,22
126,Luanda,Angola,727,815,231,22
57,Bujumbura,Burundi,728,811,111,22


Saved city-level statistical outlier analysis.


In [16]:
# Temporal Continuity + Multivariate Consistency Repair

df_temporal = df_stat.copy()
df_temporal = df_temporal.sort_values(["location_name", "last_updated"]).reset_index(drop=True)

# Per-hour sudden change thresholds.
# These are conservative thresholds for detecting likely data glitches.

temporal_rate_thresholds = {
    "temperature_celsius": 12,
    "feels_like_celsius": 16,
    "temperature_fahrenheit": 22,
    "feels_like_fahrenheit": 29,
    "humidity": 40,
    "cloud": 70,
    "pressure_mb": 20,
    "pressure_in": 0.6,
    "wind_kph": 80,
    "wind_mph": 50,
    "gust_kph": 110,
    "gust_mph": 70,
    "visibility_km": 25,
    "visibility_miles": 15,
    "uv_index": 6,
    "precip_mm": 60,
    "precip_in": 2.4,
    "air_quality_PM2.5": 120,
    "air_quality_PM10": 160,
    "air_quality_Carbon_Monoxide": 2000,
    "air_quality_Ozone": 120,
    "air_quality_Nitrogen_dioxide": 120,
    "air_quality_Sulphur_dioxide": 80,
}

temporal_features = [
    col for col in temporal_rate_thresholds.keys()
    if col in df_temporal.columns
]

print("Temporal features used:")
print(temporal_features)

repair_records = []
for city, g in df_temporal.groupby("location_name"):
    g = g.sort_values("last_updated").copy()
    if len(g) < 5:
        continue
    city_idx = g.index
    time_diff_hours = g["last_updated"].diff().dt.total_seconds() / 3600
    time_diff_hours = time_diff_hours.replace(0, np.nan)
    temporal_jump_matrix = pd.DataFrame(False, index=city_idx, columns=temporal_features)

    # First-order derivative
    for col in temporal_features:
        threshold = temporal_rate_thresholds[col]

        values = pd.to_numeric(g[col], errors="coerce")
        rate_change = values.diff().abs() / time_diff_hours

        temporal_jump_matrix[col] = rate_change > threshold

    jump_feature_count = temporal_jump_matrix.sum(axis=1)

    # Multiple features jump together: more likely real extreme weather
    multivariate_extreme_mask = jump_feature_count > 5

    # Only 1-3 features jump: more suspicious
    sparse_jump_mask = (jump_feature_count >= 1) & (jump_feature_count <= 5)

    # Decide which values should be repaired
    for idx in city_idx:
        if not sparse_jump_mask.loc[idx]:
            continue

        if multivariate_extreme_mask.loc[idx]:
            continue

        jumped_features = temporal_jump_matrix.columns[
            temporal_jump_matrix.loc[idx]
        ].tolist()

        for col in jumped_features:
            iqr_col = f"{col}_iqr_outlier"
            z_col = f"{col}_zscore_outlier"

            iqr_support = bool(df_temporal.loc[idx, iqr_col]) if iqr_col in df_temporal.columns else False
            z_support = bool(df_temporal.loc[idx, z_col]) if z_col in df_temporal.columns else False
            iso_support = bool(df_temporal.loc[idx, "isolation_forest_outlier"])

            # User requirement:
            # If any one of IQR, Z-score, or Isolation Forest supports anomaly,
            # and the jump is sparse, treat it as suspicious data error.
            # has_statistical_support = iqr_support or z_support or iso_support
            support_count = int(iqr_support) + int(z_support) + int(iso_support)
            has_statistical_support = support_count >= 2

            if not has_statistical_support:
                continue

            original_value = df_temporal.loc[idx, col]

            df_temporal.loc[idx, col] = np.nan

            repair_records.append({
                "country": df_temporal.loc[idx, "country"],
                "location_name": city,
                "last_updated": df_temporal.loc[idx, "last_updated"],
                "feature": col,
                "original_value": original_value,
                "jump_feature_count": int(jump_feature_count.loc[idx]),
                "iqr_support": iqr_support,
                "zscore_support": z_support,
                "isolation_forest_support": iso_support,
                "reason": "Sparse temporal jump with statistical anomaly support"
            })

temporal_repair_log_before = pd.DataFrame(repair_records)

display(temporal_repair_log_before.head(30))
print("Candidate suspicious values to repair:", len(temporal_repair_log_before))

temporal_repair_log_before.to_csv(
    os.path.join(REPORT_DIR, "temporal_repair_log_before_interpolation.csv"),
    index=False
)

Temporal features used:
['temperature_celsius', 'feels_like_celsius', 'temperature_fahrenheit', 'feels_like_fahrenheit', 'humidity', 'cloud', 'pressure_mb', 'pressure_in', 'wind_kph', 'wind_mph', 'gust_kph', 'gust_mph', 'visibility_km', 'visibility_miles', 'uv_index', 'precip_mm', 'precip_in', 'air_quality_PM2.5', 'air_quality_PM10', 'air_quality_Carbon_Monoxide', 'air_quality_Ozone', 'air_quality_Nitrogen_dioxide', 'air_quality_Sulphur_dioxide']


,country,location_name,last_updated,feature,original_value,jump_feature_count,iqr_support,zscore_support,isolation_forest_support,reason
0,Burundi,Bujumbura,2024-06-23 15:45:00,wind_kph,2963.200,4,True,True,True,Sparse temporal jump with statistical anomaly ...
1,Burundi,Bujumbura,2024-06-23 15:45:00,wind_mph,1841.200,4,True,True,True,Sparse temporal jump with statistical anomaly ...
2,Burundi,Bujumbura,2024-06-23 15:45:00,gust_kph,2970.400,4,True,True,True,Sparse temporal jump with statistical anomaly ...
3,Burundi,Bujumbura,2024-06-23 15:45:00,gust_mph,1845.700,4,True,True,True,Sparse temporal jump with statistical anomaly ...
4,Indonesia,Jakarta,2024-05-16 21:00:00,air_quality_Carbon_Monoxide,19653.301,1,True,False,True,Sparse temporal jump with statistical anomaly ...
5,Saudi Arabia,Riyadh,2024-11-28 14:00:00,air_quality_PM10,5554.810,1,True,True,False,Sparse temporal jump with statistical anomaly ...
6,Saudi Arabia,Riyadh,2024-12-10 13:45:00,air_quality_PM10,5224.030,1,True,True,False,Sparse temporal jump with statistical anomaly ...
7,Honduras,Tegucigalpa,2025-01-28 05:00:00,pressure_mb,3006.000,2,True,True,False,Sparse temporal jump with statistical anomaly ...
8,Honduras,Tegucigalpa,2025-01-28 05:00:00,pressure_in,88.770,2,True,True,False,Sparse temporal jump with statistical anomaly ...
9,Iran,Tehran,2025-02-09 14:00:00,pressure_mb,3000.000,2,True,True,True,Sparse temporal jump with statistical anomaly ...


Candidate suspicious values to repair: 11


In [12]:
# Interpolate Suspicious Temporal Outliers
df_before_temporal_interpolation = df_temporal.copy()
df_temporal_repaired = (
    df_temporal
    .groupby("location_name", group_keys=False)
    .apply(lambda g: interpolate_city_numeric(g, temporal_features, max_gap=3))
)

# Build final repair log: original value -> repaired value
final_temporal_repair_records = []
if len(temporal_repair_log_before) > 0:
    lookup_cols = ["country", "location_name", "last_updated"] + temporal_features
    repaired_lookup = df_temporal_repaired[lookup_cols].copy()

    for _, row in temporal_repair_log_before.iterrows():
        country = row["country"]
        city = row["location_name"]
        timestamp = row["last_updated"]
        feature = row["feature"]
        original_value = row["original_value"]
        matched = repaired_lookup[
            (repaired_lookup["country"] == country) &
            (repaired_lookup["location_name"] == city) &
            (repaired_lookup["last_updated"] == timestamp)
        ]

        if len(matched) == 0:
            repaired_value = np.nan
            status = "row_not_found"
        else:
            repaired_value = matched.iloc[0][feature]
            if pd.isna(repaired_value):
                status = "not_repaired_remaining_nan"
            else:
                status = "repaired_by_city_time_interpolation"

        record = row.to_dict()
        record["repaired_value"] = repaired_value
        record["repair_status"] = status
        final_temporal_repair_records.append(record)

temporal_repair_log_after = pd.DataFrame(final_temporal_repair_records)
display(temporal_repair_log_after.head(30))
temporal_repair_log_after.to_csv(
    os.path.join(REPORT_DIR, "temporal_repair_log_after_interpolation.csv"),
    index=False
)

if len(temporal_repair_log_after) > 0:
    temporal_repair_summary = pd.DataFrame([{
        "candidate_values_marked_for_repair": len(temporal_repair_log_after),
        "successfully_repaired_values": int((temporal_repair_log_after["repair_status"] == "repaired_by_city_time_interpolation").sum()),
        "unrepaired_values_remaining_nan": int((temporal_repair_log_after["repair_status"] != "repaired_by_city_time_interpolation").sum()),
        "modified_rows": temporal_repair_log_after[
            temporal_repair_log_after["repair_status"] == "repaired_by_city_time_interpolation"
        ][["country", "location_name", "last_updated"]].drop_duplicates().shape[0]
    }])
else:
    temporal_repair_summary = pd.DataFrame([{
        "candidate_values_marked_for_repair": 0,
        "successfully_repaired_values": 0,
        "unrepaired_values_remaining_nan": 0,
        "modified_rows": 0
    }])

display(temporal_repair_summary)
temporal_repair_summary.to_csv(
    os.path.join(REPORT_DIR, "temporal_repair_summary.csv"),
    index=False
)

,country,location_name,last_updated,feature,original_value,jump_feature_count,iqr_support,zscore_support,isolation_forest_support,reason,repaired_value,repair_status
0,Burundi,Bujumbura,2024-06-23 15:45:00,wind_kph,2963.200,4,True,True,True,Sparse temporal jump with statistical anomaly ...,27.550000,repaired_by_city_time_interpolation
1,Burundi,Bujumbura,2024-06-23 15:45:00,wind_mph,1841.200,4,True,True,True,Sparse temporal jump with statistical anomaly ...,17.100000,repaired_by_city_time_interpolation
2,Burundi,Bujumbura,2024-06-23 15:45:00,gust_kph,2970.400,4,True,True,True,Sparse temporal jump with statistical anomaly ...,34.750000,repaired_by_city_time_interpolation
3,Burundi,Bujumbura,2024-06-23 15:45:00,gust_mph,1845.700,4,True,True,True,Sparse temporal jump with statistical anomaly ...,21.600000,repaired_by_city_time_interpolation
4,Indonesia,Jakarta,2024-05-16 21:00:00,air_quality_Carbon_Monoxide,19653.301,1,True,False,True,Sparse temporal jump with statistical anomaly ...,7015.410232,repaired_by_city_time_interpolation
5,Saudi Arabia,Riyadh,2024-11-28 14:00:00,air_quality_PM10,5554.810,1,True,True,False,Sparse temporal jump with statistical anomaly ...,2270.662250,repaired_by_city_time_interpolation
6,Saudi Arabia,Riyadh,2024-12-10 13:45:00,air_quality_PM10,5224.030,1,True,True,False,Sparse temporal jump with statistical anomaly ...,1752.014346,repaired_by_city_time_interpolation
7,Honduras,Tegucigalpa,2025-01-28 05:00:00,pressure_mb,3006.000,2,True,True,False,Sparse temporal jump with statistical anomaly ...,1019.000000,repaired_by_city_time_interpolation
8,Honduras,Tegucigalpa,2025-01-28 05:00:00,pressure_in,88.770,2,True,True,False,Sparse temporal jump with statistical anomaly ...,30.090000,repaired_by_city_time_interpolation
9,Iran,Tehran,2025-02-09 14:00:00,pressure_mb,3000.000,2,True,True,True,Sparse temporal jump with statistical anomaly ...,1019.492147,repaired_by_city_time_interpolation


,candidate_values_marked_for_repair,successfully_repaired_values,unrepaired_values_remaining_nan,modified_rows
0,11,11,0,6


In [13]:
# Save Final Outlier-Cleaned Dataset
df_outlier_cleaned = df_temporal_repaired.copy()

# Remove temporary outlier flag columns before final modeling dataset
temporary_flag_cols = (
    iqr_flag_cols
    + zscore_flag_cols
    + ["isolation_forest_outlier", "isolation_forest_score"]
)

temporary_flag_cols = [c for c in temporary_flag_cols if c in df_outlier_cleaned.columns]
df_outlier_cleaned = df_outlier_cleaned.drop(columns=temporary_flag_cols)
df_outlier_cleaned = df_outlier_cleaned.sort_values(
    ["location_name", "last_updated"]
).reset_index(drop=True)
FINAL_OUTLIER_CLEANED_FILE = os.path.join(
    OUTLIER_OUTPUT_DIR,
    "GlobalWeatherRepository_missing_outlier_cleaned.csv")

df_outlier_cleaned.to_csv(FINAL_OUTLIER_CLEANED_FILE, index=False)
print("Saved final outlier-cleaned full dataset to:")
print(FINAL_OUTLIER_CLEANED_FILE)

Saved final outlier-cleaned full dataset to:
/content/drive/MyDrive/Weather Trend Forecasting/processed_outputs/outlier_processed_outputs/GlobalWeatherRepository_missing_outlier_cleaned.csv


In [14]:
# Save final city-level outlier-cleaned files
# This overwrites / refreshes files in OUTLIER_CITY_DIR
os.makedirs(OUTLIER_CITY_DIR, exist_ok=True)
used_filenames = set()

for city, g in df_outlier_cleaned.groupby("location_name"):
    safe_city_name = clean_city_filename(city)
    city_file = os.path.join(OUTLIER_CITY_DIR, f"{safe_city_name}_outlier_cleaned.csv")
    counter = 1
    while city_file in used_filenames:
        city_file = os.path.join(OUTLIER_CITY_DIR, f"{safe_city_name}_{counter}_outlier_cleaned.csv")
        counter += 1
    used_filenames.add(city_file)
    g.sort_values("last_updated").to_csv(city_file, index=False)

print("Saved final city-level outlier-cleaned files to:")
print(OUTLIER_CITY_DIR)

Saved final city-level outlier-cleaned files to:
/content/drive/MyDrive/Weather Trend Forecasting/processed_outputs/outlier_processed_outputs/outlier_cleaned_by_city


In [15]:
# Final Outlier Processing Summary
final_summary = pd.DataFrame([{
    "rows_before_outlier_processing": len(df),
    "rows_after_outlier_processing": len(df_outlier_cleaned),
    "number_of_cities": df_outlier_cleaned["location_name"].nunique(),
    "number_of_countries": df_outlier_cleaned["country"].nunique(),
    "physically_invalid_values_detected": int(repair_summary["total_physically_invalid_values_detected"].iloc[0]),
    "physically_invalid_values_repaired": int(repair_summary["successfully_repaired_values"].iloc[0]),
    "temporal_suspicious_values_detected": int(temporal_repair_summary["candidate_values_marked_for_repair"].iloc[0]),
    "temporal_suspicious_values_repaired": int(temporal_repair_summary["successfully_repaired_values"].iloc[0]),
    "start_time": df_outlier_cleaned["last_updated"].min(),
    "end_time": df_outlier_cleaned["last_updated"].max()
}])

display(final_summary)
final_summary.to_csv(
    os.path.join(REPORT_DIR, "final_outlier_processing_summary.csv"),
    index=False
)
print("Final outlier processing completed.")

,rows_before_outlier_processing,rows_after_outlier_processing,number_of_cities,number_of_countries,physically_invalid_values_detected,physically_invalid_values_repaired,temporal_suspicious_values_detected,temporal_suspicious_values_repaired,start_time,end_time
0,141508,141508,257,211,4,4,11,11,2024-05-16 01:45:00,2026-05-15 19:45:00


Final outlier processing completed.


In [18]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

FINAL_VIS_DIR = os.path.join(
    FIGURE_DIR,
    "final_repaired_outlier_42_separate_figures"
)

os.makedirs(FINAL_VIS_DIR, exist_ok=True)

print("Figures will be saved to:")
print(FINAL_VIS_DIR)


# Prepare before / after datasets

# BEFORE repair: physical-validated data + statistical flags
df_before = df_stat.copy()

# AFTER repair: final outlier-cleaned data
df_after = df_outlier_cleaned.copy()

df_before["last_updated"] = pd.to_datetime(df_before["last_updated"], errors="coerce")
df_after["last_updated"] = pd.to_datetime(df_after["last_updated"], errors="coerce")

repair_vis_log = temporal_repair_log_after.copy()
repair_vis_log["last_updated"] = pd.to_datetime(
    repair_vis_log["last_updated"],
    errors="coerce")

repair_vis_log = repair_vis_log[
    repair_vis_log["repair_status"] == "repaired_by_city_time_interpolation"
].copy()

print("Number of repaired values to visualize:", len(repair_vis_log))
display(repair_vis_log)

Figures will be saved to:
/content/drive/MyDrive/Weather Trend Forecasting/processed_outputs/outlier_processed_outputs/outlier_figures/final_repaired_outlier_42_separate_figures
Number of repaired values to visualize: 11


,country,location_name,last_updated,feature,original_value,jump_feature_count,iqr_support,zscore_support,isolation_forest_support,reason,repaired_value,repair_status
0,Burundi,Bujumbura,2024-06-23 15:45:00,wind_kph,2963.200,4,True,True,True,Sparse temporal jump with statistical anomaly ...,27.550000,repaired_by_city_time_interpolation
1,Burundi,Bujumbura,2024-06-23 15:45:00,wind_mph,1841.200,4,True,True,True,Sparse temporal jump with statistical anomaly ...,17.100000,repaired_by_city_time_interpolation
2,Burundi,Bujumbura,2024-06-23 15:45:00,gust_kph,2970.400,4,True,True,True,Sparse temporal jump with statistical anomaly ...,34.750000,repaired_by_city_time_interpolation
3,Burundi,Bujumbura,2024-06-23 15:45:00,gust_mph,1845.700,4,True,True,True,Sparse temporal jump with statistical anomaly ...,21.600000,repaired_by_city_time_interpolation
4,Indonesia,Jakarta,2024-05-16 21:00:00,air_quality_Carbon_Monoxide,19653.301,1,True,False,True,Sparse temporal jump with statistical anomaly ...,7015.410232,repaired_by_city_time_interpolation
5,Saudi Arabia,Riyadh,2024-11-28 14:00:00,air_quality_PM10,5554.810,1,True,True,False,Sparse temporal jump with statistical anomaly ...,2270.662250,repaired_by_city_time_interpolation
6,Saudi Arabia,Riyadh,2024-12-10 13:45:00,air_quality_PM10,5224.030,1,True,True,False,Sparse temporal jump with statistical anomaly ...,1752.014346,repaired_by_city_time_interpolation
7,Honduras,Tegucigalpa,2025-01-28 05:00:00,pressure_mb,3006.000,2,True,True,False,Sparse temporal jump with statistical anomaly ...,1019.000000,repaired_by_city_time_interpolation
8,Honduras,Tegucigalpa,2025-01-28 05:00:00,pressure_in,88.770,2,True,True,False,Sparse temporal jump with statistical anomaly ...,30.090000,repaired_by_city_time_interpolation
9,Iran,Tehran,2025-02-09 14:00:00,pressure_mb,3000.000,2,True,True,True,Sparse temporal jump with statistical anomaly ...,1019.492147,repaired_by_city_time_interpolation


In [19]:
# Helper Functions
def safe_filename(text):
    text = str(text)
    text = re.sub(r"[^A-Za-z0-9_]+", "_", text)
    text = re.sub(r"_+", "_", text)
    return text.strip("_")


def get_city_data(df_data, city):
    g = df_data[df_data["location_name"] == city].copy()
    g["last_updated"] = pd.to_datetime(g["last_updated"], errors="coerce")
    g = g.sort_values("last_updated")
    return g


def compute_iqr_bounds(series):
    series = pd.to_numeric(series, errors="coerce").dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    if pd.isna(iqr) or iqr == 0:
        return q1, q3, iqr, np.nan, np.nan
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return q1, q3, iqr, lower, upper


def compute_zscore(series):
    series = pd.to_numeric(series, errors="coerce")
    mean_val = series.mean()
    std_val = series.std()
    if pd.isna(std_val) or std_val == 0:
        return pd.Series(np.nan, index=series.index)
    return (series - mean_val) / std_val


def compute_city_isolation_scores(df_data, city, feature_cols):
    g = get_city_data(df_data, city)
    available_features = []
    for col in feature_cols:
        if col in g.columns:
            s = pd.to_numeric(g[col], errors="coerce")
            if s.notna().sum() >= 30 and s.std() > 0:
                available_features.append(col)
    if len(g) < 50 or len(available_features) < 3:
        g["iforest_score"] = np.nan
        g["iforest_outlier"] = False
        return g
    X = g[available_features].copy()
    for col in available_features:
        X[col] = pd.to_numeric(X[col], errors="coerce")
    X = X.fillna(X.median(numeric_only=True))
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    model = IsolationForest(
        n_estimators=200,
        contamination=0.03,
        random_state=42,
        n_jobs=-1
    )
    pred = model.fit_predict(X_scaled)
    score = model.decision_function(X_scaled)
    g["iforest_score"] = score
    g["iforest_outlier"] = pred == -1
    return g


def get_target_value(df_data, city, timestamp, feature):
    g = get_city_data(df_data, city)
    row = g[g["last_updated"] == timestamp]
    if len(row) == 0:
        return np.nan
    return row.iloc[0][feature]

In [20]:
# IQR Single Figure
def plot_single_iqr(df_data, record, stage, save_dir):
    country = record["country"]
    city = record["location_name"]
    timestamp = record["last_updated"]
    feature = record["feature"]
    g = get_city_data(df_data, city)
    g[feature] = pd.to_numeric(g[feature], errors="coerce")
    target_value = get_target_value(df_data, city, timestamp, feature)
    q1, q3, iqr, lower, upper = compute_iqr_bounds(g[feature])
    plt.figure(figsize=(8, 6))
    plt.boxplot(g[feature].dropna(), vert=True)
    plt.scatter(
        1,
        target_value,
        s=120,
        marker="x" if stage == "before" else "o",
        label=f"{stage.capitalize()} target value"
    )

    if not pd.isna(lower):
        plt.axhline(lower, linestyle="--", linewidth=1.2, label="IQR lower bound")
        plt.axhline(upper, linestyle="--", linewidth=1.2, label="IQR upper bound")

    plt.title(
        f"{stage.capitalize()} Step 7 Repair - IQR\n"
        f"{country} | {city} | {feature}\n"
        f"Target time: {timestamp}"
    )
    plt.ylabel(feature)
    plt.xticks([1], [feature])
    plt.legend()
    plt.tight_layout()

    fname = (
        f"{safe_filename(city)}_{safe_filename(feature)}_"
        f"{safe_filename(timestamp)}_{stage}_IQR.png"
    )

    path = os.path.join(save_dir, fname)
    plt.savefig(path, dpi=300)
    plt.close()
    return path

In [21]:
# Z-score Single Figure
def plot_single_zscore(df_data, record, stage, save_dir):
    country = record["country"]
    city = record["location_name"]
    timestamp = record["last_updated"]
    feature = record["feature"]

    g = get_city_data(df_data, city)
    g[feature] = pd.to_numeric(g[feature], errors="coerce")
    g["zscore"] = compute_zscore(g[feature])

    target_row = g[g["last_updated"] == timestamp]
    plt.figure(figsize=(12, 5))
    plt.plot(
        g["last_updated"],
        g["zscore"],
        linewidth=1.5,
        label="Z-score over time"
    )

    plt.axhline(3, linestyle="--", linewidth=1.2, label="+3 threshold")
    plt.axhline(-3, linestyle="--", linewidth=1.2, label="-3 threshold")
    plt.axhline(0, linewidth=1)

    if len(target_row) > 0:
        plt.scatter(
            target_row["last_updated"],
            target_row["zscore"],
            s=120,
            marker="x" if stage == "before" else "o",
            label=f"{stage.capitalize()} target value"
        )

    plt.title(
        f"{stage.capitalize()} Step 7 Repair - Z-score\n"
        f"{country} | {city} | {feature}\n"
        f"Target time: {timestamp}"
    )
    plt.xlabel("Time")
    plt.ylabel("Z-score")
    plt.legend()
    plt.tight_layout()

    fname = (
        f"{safe_filename(city)}_{safe_filename(feature)}_"
        f"{safe_filename(timestamp)}_{stage}_Zscore.png"
    )

    path = os.path.join(save_dir, fname)
    plt.savefig(path, dpi=300)
    plt.close()
    return path

In [22]:
# Isolation Forest Single Figure
def plot_single_isolation_forest(df_data, record, stage, save_dir):
    country = record["country"]
    city = record["location_name"]
    timestamp = record["last_updated"]
    feature = record["feature"]
    g_if = compute_city_isolation_scores(df_data, city, outlier_numeric_cols)
    target_row = g_if[g_if["last_updated"] == timestamp]
    plt.figure(figsize=(12, 5))
    plt.plot(
        g_if["last_updated"],
        g_if["iforest_score"],
        linewidth=1.5,
        label="Isolation Forest decision score"
    )

    plt.axhline(
        0,
        linestyle="--",
        linewidth=1.2,
        label="Decision boundary"
    )

    if len(target_row) > 0:
        plt.scatter(
            target_row["last_updated"],
            target_row["iforest_score"],
            s=120,
            marker="x" if stage == "before" else "o",
            label=f"{stage.capitalize()} target row"
        )

    plt.title(
        f"{stage.capitalize()} Step 7 Repair - Isolation Forest\n"
        f"{country} | {city} | affected feature: {feature}\n"
        f"Target time: {timestamp}"
    )
    plt.xlabel("Time")
    plt.ylabel("Decision score")
    plt.legend()
    plt.tight_layout()

    fname = (
        f"{safe_filename(city)}_{safe_filename(feature)}_"
        f"{safe_filename(timestamp)}_{stage}_IsolationForest.png"
    )

    path = os.path.join(save_dir, fname)
    plt.savefig(path, dpi=300)
    plt.close()
    return path

In [23]:
# Visualization of Outlier Analysis
# Comparison of 7 abnormal data before and after modification

saved_figure_records = []

for idx, record in repair_vis_log.iterrows():
    print(
        f"Generating 6 figures for: "
        f"{record['location_name']} | {record['feature']} | {record['last_updated']}"
    )

    for stage, df_stage in [("before", df_before), ("after", df_after)]:

        iqr_path = plot_single_iqr(
            df_stage,
            record,
            stage,
            FINAL_VIS_DIR
        )

        zscore_path = plot_single_zscore(
            df_stage,
            record,
            stage,
            FINAL_VIS_DIR
        )

        iforest_path = plot_single_isolation_forest(
            df_stage,
            record,
            stage,
            FINAL_VIS_DIR
        )

        saved_figure_records.extend([
            {
                "country": record["country"],
                "location_name": record["location_name"],
                "last_updated": record["last_updated"],
                "feature": record["feature"],
                "stage": stage,
                "method": "IQR",
                "figure_path": iqr_path
            },
            {
                "country": record["country"],
                "location_name": record["location_name"],
                "last_updated": record["last_updated"],
                "feature": record["feature"],
                "stage": stage,
                "method": "Z-score",
                "figure_path": zscore_path
            },
            {
                "country": record["country"],
                "location_name": record["location_name"],
                "last_updated": record["last_updated"],
                "feature": record["feature"],
                "stage": stage,
                "method": "Isolation Forest",
                "figure_path": iforest_path
            }
        ])

saved_figure_summary = pd.DataFrame(saved_figure_records)

display(saved_figure_summary)

summary_path = os.path.join(
    FINAL_VIS_DIR,
    "saved_42_repaired_outlier_visualization_summary.csv"
)

saved_figure_summary.to_csv(summary_path, index=False)

print("Total figures generated:", len(saved_figure_summary))
print("Summary saved to:")
print(summary_path)
print("All figures saved to:")
print(FINAL_VIS_DIR)

Generating 6 figures for: Bujumbura | wind_kph | 2024-06-23 15:45:00
Generating 6 figures for: Bujumbura | wind_mph | 2024-06-23 15:45:00
Generating 6 figures for: Bujumbura | gust_kph | 2024-06-23 15:45:00
Generating 6 figures for: Bujumbura | gust_mph | 2024-06-23 15:45:00
Generating 6 figures for: Jakarta | air_quality_Carbon_Monoxide | 2024-05-16 21:00:00
Generating 6 figures for: Riyadh | air_quality_PM10 | 2024-11-28 14:00:00
Generating 6 figures for: Riyadh | air_quality_PM10 | 2024-12-10 13:45:00
Generating 6 figures for: Tegucigalpa | pressure_mb | 2025-01-28 05:00:00
Generating 6 figures for: Tegucigalpa | pressure_in | 2025-01-28 05:00:00
Generating 6 figures for: Tehran | pressure_mb | 2025-02-09 14:00:00
Generating 6 figures for: Tehran | pressure_in | 2025-02-09 14:00:00


,country,location_name,last_updated,feature,stage,method,figure_path
0,Burundi,Bujumbura,2024-06-23 15:45:00,wind_kph,before,IQR,/content/drive/MyDrive/Weather Trend Forecasti...
1,Burundi,Bujumbura,2024-06-23 15:45:00,wind_kph,before,Z-score,/content/drive/MyDrive/Weather Trend Forecasti...
2,Burundi,Bujumbura,2024-06-23 15:45:00,wind_kph,before,Isolation Forest,/content/drive/MyDrive/Weather Trend Forecasti...
3,Burundi,Bujumbura,2024-06-23 15:45:00,wind_kph,after,IQR,/content/drive/MyDrive/Weather Trend Forecasti...
4,Burundi,Bujumbura,2024-06-23 15:45:00,wind_kph,after,Z-score,/content/drive/MyDrive/Weather Trend Forecasti...
...,...,...,...,...,...,...,...
61,Iran,Tehran,2025-02-09 14:00:00,pressure_in,before,Z-score,/content/drive/MyDrive/Weather Trend Forecasti...
62,Iran,Tehran,2025-02-09 14:00:00,pressure_in,before,Isolation Forest,/content/drive/MyDrive/Weather Trend Forecasti...
63,Iran,Tehran,2025-02-09 14:00:00,pressure_in,after,IQR,/content/drive/MyDrive/Weather Trend Forecasti...
64,Iran,Tehran,2025-02-09 14:00:00,pressure_in,after,Z-score,/content/drive/MyDrive/Weather Trend Forecasti...


Total figures generated: 66
Summary saved to:
/content/drive/MyDrive/Weather Trend Forecasting/processed_outputs/outlier_processed_outputs/outlier_figures/final_repaired_outlier_42_separate_figures/saved_42_repaired_outlier_visualization_summary.csv
All figures saved to:
/content/drive/MyDrive/Weather Trend Forecasting/processed_outputs/outlier_processed_outputs/outlier_figures/final_repaired_outlier_42_separate_figures
